<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.2: Similitud de Coseno e Indice Invertido

 > **Prof. Heider Sanchez**
 
 > **Grupo:** Juan Ticlia, Paulo Miranda, Jose Huaman 

## Introducción

Este laboratorio extiende las capacidades desarrolladas en el laboratorio 7.1, enfocándose en técnicas avanzadas de recuperación de información. Utilizaremos los Bag of Words previamente generados para implementar dos funcionalidades esenciales en los motores de búsqueda modernos: el **Índice Invertido** para recuperación eficiente de documentos, y la **Similitud de Coseno** para resultados ordenados por relevancia.


### Objetivos
- Implementar la conexión a PostgreSQL para extraer id, contenido y bag_of_words de los documentos.
- Construir el índice invertido (posting lists), calcular estadísticas IDF y la norma vectorial de cada documento.
- Implementar búsquedas booleanas (AND, OR y AND-NOT) con complejidad O(n+m) usando el índice invertido.
- Implementar búsquedas rankeadas usando la Similitud de Coseno:
  - Procesar consultas en lenguaje natural aplicando tokenización y cálculo del TF.
  - Utilizar el índice invertido para obtener los documentos que intersectan con la query.
  - Calcular similitud de coseno con TF-IDF entre la query y los documentos recuperados.
  - Devolver los top-k resultados ordenados por relevancia.
  - **Evitar usar la representación vectorial dispersa.**

In [1]:
import psycopg2
import pandas as pd

"""
def connect_db():
    conn = psycopg2.connect(
        dbname="<DB>",
        user="<USER>",
        password="<PASSWORD>",
        host="<HOST>"
    )
    return conn
"""

def connect_db():
    try:
        conn = psycopg2.connect(
            dbname="midb",
            user="postgres",
            password="postgres",
            host="localhost",
            port="5433",
            options=f"-c search_path=lab08"
            # las tablas estaban en el esquema lab08, por eso esta como patch de busqueda para las tablas
        )
        return conn

    except psycopg2.Error as e:
        print("Error al conectar a PostgreSQL:")
        print(e)
        return None


def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido, bag_of_words FROM noticias;"
    df = pd.read_sql(query, conn)
    # El atributo está como jsonb en la db, cuadno llega a python no es necesario castearlo (da error)
    # por eso estoy comentando la linea que sigue:
    # df['bag_of_words'] = df['bag_of_words'].apply(json.loads)
    conn.close()
    return df

noticias_df = fetch_data()

ipykernel_16424\3262723245.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## 1. (6 puntos) Construcción del Indice Invertido 

A partir de los  `bag of words` almacenados en la base de datos  (Laboratorio 8.1), se debe construir un índice invertido y conservarlo en un diccionario de Python para su eficiente recuperación.

In [2]:
# Preproceso de un texto de lab 8.1
import nltk
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
# nltk.download('punkt')
# nltk.download('punkt_tab')
def fetch_stopwords():
    conn = connect_db()
    query = "SELECT word FROM stopwords;"
    df = pd.read_sql(query, conn)
    conn.close()
    stopwords_set = set(df["word"].str.lower())
    return stopwords_set
stemmer = SnowballStemmer("spanish")
stopwords = fetch_stopwords()
def preprocess(text):
    text = text.lower()
    text = "".join(c for c in text if c.isalpha() or c.isspace())
    tokens = word_tokenize(text, language='spanish')
    words = [t for t in tokens if t not in stopwords]
    words = [stemmer.stem(w) for w in words]
    return words

ipykernel_16424\2800580072.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [3]:
import math

class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.idf = {}
        self.length = {}
        N = 0

    def build_from_db(self):
        # Leer desde PostgreSQL todos los bag of words
        # Construir el índice invertido, el idf y la norma (longitud) de cada documento
        
        """
        index  = {
            "word1": [("doc1", tf1), ("doc2", tf2), ("doc3", tf3)],
            "word2": [("doc2", tf2), ("doc4", tf4)],
            "word3": [("doc3", tf3), ("doc5", tf5)],
        } 
        idf  = {
            "word1": 3,
            "word2": 2,
            "word3": 2,
        } 
        length = {
            "doc1": 15.5236,
            "doc2": 10.5236,
            "doc3": 5.5236,
        }
        """

        noticias_df = fetch_data()
        # noticias_df = noticias_df.head(100)


        # INDEX: Construccion del indice invertido index = [(id, tf), ...]

        # Aclaracion. tf no normalizado
        
        for row in noticias_df.itertuples():
            doc_id = row.id
            bow = row.bag_of_words

            # Guardar en indice invertido
            for word in bow:
                tupla = (doc_id, bow[word])
                if word not in self.index:
                    self.index[word] = []
                self.index[word].append(tupla)

        # Ordenar cada lista de documentos correspondiente a cada palabra
        # para que al hacer las querys sea eficiente por merge
        for word in self.index:
            self.index[word].sort(key=lambda x: x[0])

        

        # IDF: Construccion del ranked retrieval idf

        # Aclaracion. Se usa log natural (base e)

        N = len(noticias_df)

        for word in self.index:
            df = len(self.index[word])
            self.idf[word] = math.log(N / df)

        

        # LENGTH: Construccion de la norma weight tf.idf de cada documento
        # Aclaracion. TF amortizado | tf = log_natural(1 + tf)

        for row in noticias_df.itertuples():
            doc_id = row.id
            bow = row.bag_of_words

            length = 0

            for word, tf in bow.items():
                tf = math.log(1 + tf)
                tfidf = tf * self.idf[word]
                length += tfidf ** 2

            self.length[doc_id] = math.sqrt(length)
    


    def L(self, word):
        # Retorna la lista de documentos que contienen a word
        tokens = preprocess(word)
        if not tokens:
            return []
        return self.index.get(tokens[0], [])

    def showDocuments(self, result):
        # Extraemos solo el doc_id
        doc_ids = [doc_id for doc_id, tf in result]

        # print("Limite de 10 doc: ")
        # doc_ids = doc_ids[:10]

        return doc_ids



    def cosine_search(self, query, top_k=5):  
        score = {}
        # No es necesario usar vectores numericos del tamaño del vocabulario
        # Guiarse del algoritmo visto en clase
        # Se debe calcular el tf-idf de la query y de cada documento
        # La query se debe procesar  al igual como se procesaron los documentos (bag of words)

        query_words = preprocess(query)
        query_bow = {}
        for word in query_words:
            query_bow[word] = query_bow.get(word, 0) + 1
            
        for term, tf_q in query_bow.items():
            # sin esto no funcionaba...
            if term not in self.idf:
                continue 
                
            w_t_q = math.log(1 + tf_q) * self.idf[term]
            
            postings_list = self.L(term) # Lista de tuplas

            for doc_id, tf_d in postings_list:

                w_t_d = math.log(1 + tf_d) * self.idf[term]
                
                if doc_id not in score:
                    score[doc_id] = 0.0
                score[doc_id] += (w_t_q * w_t_d)
                
        for doc_id in score:
            if self.length[doc_id] > 0:
                score[doc_id] = score[doc_id] / self.length[doc_id]

        result = sorted(score.items(), key= lambda tup: tup[1], reverse=True)
        # retornamos los k documentos mas relevantes (de mayor similitud a la query)
        return result[:top_k]
    


## 2. (6 puntos) Consultas Booleanas usando el indice invertido

Implementar búsquedas booleanas utilizando el índice invertido construido anteriormente. La búsqueda debe:

- Soportar los operadores básicos:
    - AND: intersección de documentos
    - OR: unión de documentos
    - AND-NOT: diferencia de documentos
- Procesar consultas como:
    - "sostenibilidad AND ambiente AND renovable"
    - "tecnología AND (banca OR finanzas)"
    - "economía AND-NOT inflación"    

####  Pruebas funcionales

In [4]:
idx = InvertedIndex()
idx.build_from_db()

def AND(list1, list2):
    # Implementar la intersección de dos listas O(n +m)
    # Si alguna de las listas está vacía, no hay intersección posible
    if not list1 or not list2:
        return []
    
    n, m = len(list1), len(list2)
    p1, p2 = 0, 0
    answer = []
    while p1 < n and p2 < m:
        doc_id1, tf1 = list1[p1]
        doc_id2, tf2 = list2[p2]
        if doc_id1 == doc_id2:
            answer.append((doc_id1, tf1 + tf2))
            p1 += 1
            p2 += 1
        elif doc_id1 < doc_id2:
            p1 += 1
        else:
            p2 += 1
    return answer

def OR(list1, list2):
    # Implementar la unión de dos listas O(n +m)
    # listas vacias
    if not list1: return list2
    if not list2: return list1
    
    n, m = len(list1), len(list2)
    p1, p2 = 0, 0
    answer = []
    while p1 < n and p2 < m:
        doc_id1, tf1 = list1[p1]
        doc_id2, tf2 = list2[p2]
        if doc_id1 == doc_id2:
            answer.append((doc_id1, tf1 + tf2))
            p1 += 1
            p2 += 1
        elif doc_id1 < doc_id2:
            answer.append(list1[p1])
            p1 += 1
        else:
            answer.append(list2[p2]) 
            p2 += 1
            
    # Vaciar los elementos restantes
    while p1 < n:
        answer.append(list1[p1])
        p1 += 1
    while p2 < m:
        answer.append(list2[p2])
        p2 += 1
        
    return answer

def AND_NOT(list1, list2):
    # Implementar la diferencia de dos listas O(n +m)
    n, m = len(list1), len(list2)
    p1, p2 = 0, 0
    answer = []
    while p1 < n and p2 < m:
        doc_id1, tf1 = list1[p1]
        doc_id2, tf2 = list2[p2]
        if doc_id1 == doc_id2:
            # Si está en ambos, se excluye
            p1 += 1
            p2 += 1
        elif doc_id1 < doc_id2:
            answer.append(list1[p1])
            p1 += 1
        else:
            p2 += 1
            
    # Si list2 se acabó, todo lo que quede en list1 NO está en list2
    while p1 < n:
        answer.append(list1[p1])
        p1 += 1
        
    return answer

# Prueba 1
print("Prueba 1:", "-"*10)
result = AND(idx.L("sostenibilidad"), AND(idx.L("ambiente"), idx.L("renovables")))
print("sostenibilidad AND ambiente AND renovable:\n    doc_id -> ", idx.showDocuments(result))

# Prueba 2
print("Prueba 2:", "-"*10)
result = AND(idx.L("tecnología"), OR(idx.L("banca"), idx.L("finanzas")))
print("tecnología AND (banca OR finanzas):\n    doc_id -> ", idx.showDocuments(result))

# Prueba 3
print("Prueba 3:", "-"*10)
result = AND_NOT(idx.L("economía"), idx.L("inflación"))
print("economía AND-NOT inflación:\n    doc_id -> " , idx.showDocuments(result))

#Agregar dos pruebas mas combinando los operadores AND, OR, AND_NOT
# Prueba 4
print("Prueba 4:", "-"*10)
result = AND_NOT(AND(OR(idx.L("renovable"), idx.L("eléctrico")), idx.L("energía")), idx.L("contaminación"))
print("(renovable OR eléctrico) AND energía AND-NOT contaminación:\n    doc_id -> ", idx.showDocuments(result))

# Prueba 5
print("Prueba 5:", "-"*10)
result = AND_NOT(OR(idx.L("economía"), idx.L("mercado")),AND(idx.L("financiero"), idx.L("pérdidas")))
print("(economía OR mercado) AND-NOT (financiero AND pérdidas):\n    doc_id -> ", idx.showDocuments(result))

ipykernel_16424\3262723245.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Prueba 1: ----------
sostenibilidad AND ambiente AND renovable:
    doc_id ->  [52, 397, 420, 469, 637, 1126, 1165, 1179]
Prueba 2: ----------
tecnología AND (banca OR finanzas):
    doc_id ->  [12, 13, 15, 16, 17, 20, 30, 49, 52, 55, 63, 78, 85, 88, 93, 111, 116, 135, 144, 147, 156, 178, 179, 187, 193, 194, 205, 206, 210, 212, 226, 228, 234, 237, 246, 253, 258, 259, 269, 282, 286, 293, 302, 312, 321, 332, 358, 362, 365, 369, 376, 391, 394, 406, 415, 416, 417, 419, 430, 445, 455, 456, 460, 465, 478, 498, 507, 511, 522, 540, 543, 571, 573, 583, 584, 594, 598, 601, 606, 607, 611, 617, 623, 626, 628, 633, 639, 642, 644, 655, 665, 669, 674, 675, 679, 686, 689, 696, 705, 719, 723, 738, 748, 781, 786, 793, 819, 826, 829, 840, 842, 849, 850, 852, 864, 888, 889, 892, 895, 901, 905, 911, 916, 917, 918, 943, 951, 958, 979, 988, 992, 996, 1009, 1011, 1015, 1023, 1024, 1033, 1034, 1035, 1042, 1048, 1050, 1052, 1056, 1062, 1065, 1067, 1069, 1079, 1094, 1095, 1096, 1097, 1108, 1113, 1115, 1117, 1121

## 3. (8 puntos) Similitud de Coseno usando el indice invertido
Implementar búsqueda por similitud de coseno aprovechando el índice invertido:

- Proceso de búsqueda:
    - Recibe una consulta en lenguaje natural y un parámetro top_k
    - Utiliza el índice invertido para identificar documentos candidatos
    - Calcula similitud de coseno solo con los documentos relevantes utilizando los pesos TF-IDF
    - Retorna los top-k documentos más similares

<img src="https://1drv.ms/i/c/0c2923df9f1f816f/IQSELMi5qcbqS7lsy5sn8ZLpAZ3G2ciXdabecVJ0vhKoL78" width="500" align="" />

####  Pruebas funcionales

In [5]:
idx = InvertedIndex()
idx.build_from_db()

test_queries = [
    { 'query': "¿Cuáles son las últimas innovaciones en la banca digital y la tecnología financiera?", 'top_k': 5 },
    { 'query': "evolución de la inflación y el crecimiento de la economía en los últimos años", 'top_k': 5 },
    { 'query': "avances sobre sostenibilidad y energías renovables para el medio ambiente", 'top_k': 5 }
]

for test in test_queries:    
    results = idx.cosine_search(test['query'], test['top_k'])
    print(f"Top {test['top_k']} documentos más similares:")    
    for doc_id, score in results:
        print(f"Doc {doc_id}  |  Score: {score:.3f}")
    print("-" * 14)

Top 5 documentos más similares:
Doc 1097  |  Score: 0.347
Doc 829  |  Score: 0.345
Doc 282  |  Score: 0.340
Doc 390  |  Score: 0.327
Doc 220  |  Score: 0.305
--------------
Top 5 documentos más similares:
Doc 58  |  Score: 0.413
Doc 352  |  Score: 0.413
Doc 986  |  Score: 0.399
Doc 62  |  Score: 0.335
Doc 487  |  Score: 0.334
--------------
Top 5 documentos más similares:
Doc 661  |  Score: 0.687
Doc 433  |  Score: 0.613
Doc 726  |  Score: 0.533
Doc 340  |  Score: 0.508
Doc 77  |  Score: 0.463
--------------


ipykernel_16424\3262723245.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
